In [ ]:
"""
Single-Node Agentic RAG System with LangGraph
All logic (retrieval, analysis, fallback, clarification) happens inside answer node.
"""
import os
import operator
import asyncio
from typing import TypedDict, Annotated, Sequence

from dotenv import load_dotenv
from decouple import config
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver



In [ ]:


# Load environment variables
load_dotenv()
DB_URL = config("DB_URL")


# ==================== STATE DEFINITION ====================
class AgentState(TypedDict):
    """State for the agent workflow"""
    messages: Annotated[Sequence[BaseMessage], operator.add]
    context: str
    next_action: str
    user_name: str


# ==================== CONTEXT RETRIEVAL ====================
class ContextRetriever:
    """Fetches context from NeonDB (single table: context_store)"""

    def __init__(self, db_url: str):
        self.db_url = db_url

    async def retrieve_context(self) -> str:
        import asyncpg
        try:
            conn = await asyncpg.connect(self.db_url)

            row = await conn.fetchrow("SELECT content FROM context_store WHERE id = 1;")

            await conn.close()

            if row and row["content"]:
                print(f"✅ Retrieved {len(row['content'])} chars from context_store")
                return row["content"]

            print("⚠️  No content found in context_store")
            return ""

        except Exception as e:
            print(f"❌ Error retrieving context: {e}")
            return ""


# ==================== SINGLE NODE RAG AGENT ====================
class RAGAgent:
    """Single-node RAG agent — all logic inside answer_node"""

    def __init__(self):
        self.llm = ChatGoogleGenerativeAI(
            model="gemini-2.0-flash",
            temperature=0.7,
            max_tokens=None,
            max_retries=2,
            api_key=config("GEMINI_API_KEY"),
        )
        self.context_retriever = ContextRetriever(DB_URL)


    async def answer_node(self, state: AgentState) -> AgentState:
        print("💬 [ANSWER NODE] Single-node RAG pipeline executing...")

        messages = state["messages"]
        question = messages[-1].content if messages else ""
        user_name = state.get("user_name", "")

        # Retrieve context right here
        context = await self.context_retriever.retrieve_context()

        # Unified prompt — with escaped braces
        answer_prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                (
                    "You are a helpful assistant that answers questions based on the following document about LLM Guardrails:\n\n"
                    "{context}\n\n"
                    "Please keep your answers concise (no more than 5 sentences) and ensure your answers "
                    "are relevant to the user's question. If you don't know the answer, say 'I don't know'."
                    """Rules:
                        - Never answer anything outside Guardrails.
                        - Never hallucinate missing facts.
                        - Use short, clear sentences.
                        - Use markdown formatting.
                    """
                ),
            ),
            ("human", "{question}"),
        ]
    )
        

        chain = answer_prompt | self.llm

        history = messages[:-1][-5:] if len(messages) > 1 else []

        result = chain.invoke({
            "context": context,
            "history": history,
            "question": question
        })

        return {
            **state,
            "messages": [AIMessage(content=result.content)],
            "next_action": "end"
        }


def create_rag_graph():
    """Build minimalist workflow with only ANSWER node."""
    
    agent = RAGAgent()

    workflow = StateGraph(AgentState)

    # Only ONE node
    workflow.add_node("answer", agent.answer_node)

    # Entry point
    workflow.set_entry_point("answer")

    # End
    workflow.add_edge("answer", END)

    checkpointer = MemorySaver()
    app = workflow.compile(checkpointer=checkpointer)

    return app





In [40]:

# ==================== IN-MEMORY CONVERSATION STORE ====================
from collections import defaultdict

class ConversationStore:
    """Stores conversation history per user/thread in memory"""
    
    def __init__(self):
        # Structure: {thread_id: [{"role": "user/assistant", "content": "..."}]}
        self.conversations = defaultdict(list)
    
    def add_message(self, thread_id: str, role: str, content: str):
        """Add a message to conversation history"""
        self.conversations[thread_id].append({
            "role": role,
            "content": content
        })
        print(f"📝 Added {role} message to thread {thread_id}")
    
    def get_messages(self, thread_id: str) -> list:
        """Retrieve all messages for a thread"""
        return self.conversations[thread_id]
    
    def get_langchain_messages(self, thread_id: str) -> list:
        """Convert stored messages to LangChain message objects"""
        messages = []
        for msg in self.conversations[thread_id]:
            if msg["role"] == "user":
                messages.append(HumanMessage(content=msg["content"]))
            elif msg["role"] == "assistant":
                messages.append(AIMessage(content=msg["content"]))
        return messages
    
    def clear_thread(self, thread_id: str):
        """Clear conversation history for a thread"""
        if thread_id in self.conversations:
            del self.conversations[thread_id]
            print(f"🗑️  Cleared thread {thread_id}")
    
    def get_thread_count(self, thread_id: str) -> int:
        """Get message count for a thread"""
        return len(self.conversations[thread_id])


# Global conversation store
conversation_store = ConversationStore()

In [41]:
# ==================== MAIN CALL FUNCTION ====================
async def chat(question: str, thread_id: str = "default") -> dict:
    """
    Main interface for the single-node RAG system.
    Backend manages conversation history automatically.
    
    Args:
        question: Current user question
        thread_id: Unique identifier for conversation thread (use user_id or session_id)
    
    Returns:
        dict: {"answer": str, "message_count": int}
    """
    
    # Retrieve conversation history from backend store
    history_messages = conversation_store.get_langchain_messages(thread_id)
    
    # Add current user question
    history_messages.append(HumanMessage(content=question))
    
    print(f"📚 Thread {thread_id} has {len(history_messages)} messages (including current)")
    
    # Create and run the graph
    app = create_rag_graph()

    config = {
        "configurable": {
            "thread_id": thread_id
        }
    }

    initial_state = {
        "messages": history_messages,
        "context": "",
        "next_action": ""
    }

    result = await app.ainvoke(initial_state, config)
    
    answer = result["messages"][-1].content
    
    # Store both user question and assistant answer in backend
    conversation_store.add_message(thread_id, "user", question)
    conversation_store.add_message(thread_id, "assistant", answer)
    
    return {
        "answer": answer,
        "message_count": conversation_store.get_thread_count(thread_id)
    }


# ==================== UTILITY FUNCTIONS ====================
async def get_conversation_history(thread_id: str) -> list:
    """Get full conversation history for a thread"""
    return conversation_store.get_messages(thread_id)


async def clear_conversation(thread_id: str):
    """Clear conversation history for a thread"""
    conversation_store.clear_thread(thread_id)




In [48]:
# ==================== EXAMPLE USAGE ====================
# Simulate a multi-turn conversation
thread = "user_123"

print("\n" + "="*50)
print("Turn 1:")
response1 = await chat("What are guardrails?", thread)
print(f"Answer: {response1['answer']}")
print(f"Messages in thread: {response1['message_count']}")





Turn 1:
📚 Thread user_123 has 7 messages (including current)
💬 [ANSWER NODE] Single-node RAG pipeline executing...
✅ Retrieved 7304 chars from context_store
📝 Added user message to thread user_123
📝 Added assistant message to thread user_123
Answer: Guardrails are algorithms that filter the inputs and outputs of trained LLMs. They determine if and how enforcement actions can be taken to reduce the risks embedded in the objects. For example, they can stop the input from being processed or adapt the output to be harmless. Guardrails identify potential misuse in the query stage and prevent the model from providing inappropriate answers.
Messages in thread: 8


In [49]:
print("\n" + "="*50)
print("Turn 2:")
response2 = await chat("Can you give me an example?", thread)
print(f"Answer: {response2['answer']}")
print(f"Messages in thread: {response2['message_count']}")




Turn 2:
📚 Thread user_123 has 9 messages (including current)
💬 [ANSWER NODE] Single-node RAG pipeline executing...
✅ Retrieved 7304 chars from context_store
📝 Added user message to thread user_123
📝 Added assistant message to thread user_123
Answer: A guardrail may stop an input to an LLM related to child exploitation from being processed or adapt the output to be harmless. Guardrails identify potential misuse in the query stage. They prevent the model from providing answers that should not be given.
Messages in thread: 10


In [50]:
print("\n" + "="*50)
print("Turn 3:")
response3 = await chat("Tell me more about the first point", thread)
print(f"Answer: {response3['answer']}")
print(f"Messages in thread: {response3['message_count']}")




Turn 3:
📚 Thread user_123 has 11 messages (including current)
💬 [ANSWER NODE] Single-node RAG pipeline executing...
✅ Retrieved 7304 chars from context_store
📝 Added user message to thread user_123
📝 Added assistant message to thread user_123
Answer: The first point refers to the requirement that guardrails should ensure LLMs are free from unintended responses, such as offensive and hate speech. This is a key aspect of ensuring ethical and safe use of LLMs.
Messages in thread: 12


In [51]:
print("\n" + "="*50)
print("Turn 3:")
response3 = await chat("My name is chizzy. If neccessary, give me a mention while responding", thread)
print(f"Answer: {response3['answer']}")
print(f"Messages in thread: {response3['message_count']}")




Turn 3:
📚 Thread user_123 has 13 messages (including current)
💬 [ANSWER NODE] Single-node RAG pipeline executing...
✅ Retrieved 7304 chars from context_store
📝 Added user message to thread user_123
📝 Added assistant message to thread user_123
Answer: I am designed to answer questions based on the provided document about LLM Guardrails. I cannot process personal information or engage in conversation outside of that context.
Messages in thread: 14


In [45]:
print("\n" + "="*50)
print("Full Conversation History:")
history = await get_conversation_history(thread)
for i, msg in enumerate(history, 1):
    print(f"{i}. [{msg['role']}]: {msg['content'][:100]}...")


Full Conversation History:
1. [user]: What are guardrails?...
2. [assistant]: Guardrails are algorithms that filter the inputs and outputs of Large Language Models (LLMs) to redu...
3. [user]: Can you give me an example?...
4. [assistant]: A guardrail may stop an input related to child exploitation from being processed by the LLMs or adap...
5. [user]: Tell me more about the first point...
6. [assistant]: The first point refers to the requirement that guardrails should ensure LLMs are free from unintende...


In [46]:
await chat("What are LLM Guardrails?")

📚 Thread default has 1 messages (including current)
💬 [ANSWER NODE] Single-node RAG pipeline executing...
✅ Retrieved 7304 chars from context_store
📝 Added user message to thread default
📝 Added assistant message to thread default


{'answer': 'LLM Guardrails are algorithms that filter the inputs and outputs of Large Language Models (LLMs). They determine if and how enforcement actions can be taken to reduce risks. For example, guardrails can stop problematic inputs or adapt outputs to be harmless. They identify potential misuse and prevent models from providing undesirable responses.',
 'message_count': 2}

In [47]:
await chat("what are the best praxctices for llms?")

📚 Thread default has 3 messages (including current)
💬 [ANSWER NODE] Single-node RAG pipeline executing...
✅ Retrieved 7304 chars from context_store
📝 Added user message to thread default
📝 Added assistant message to thread default


{'answer': 'A systematic process covering the development cycle is required to carefully build guardrails for LLMs. This includes specification, design, implementation, integration, verification, validation, and production release. A multi-disciplinary approach is called for, to precisely define requirements and corresponding metrics. A plausible design of the guardrail is neural-symbolic, with learning agents and symbolic agents collaborating in processing both the inputs and the outputs of LLMs.',
 'message_count': 4}